## **BERT Question Answering**

In [49]:
!pip install -U datasets

In this Demo we use SQuAD dataset and finetune Bert base model for interactive qeustion answering.

In [50]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    default_data_collator
)
from datasets import load_dataset
import numpy as np
from collections import OrderedDict
import warnings
warnings.filterwarnings('ignore')


Initilize BERT Model and Tokenizer

In [51]:
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForQuestionAnswering(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, 

In [52]:
print(f"Using device: {device}")
print(f"Model loaded: {model_name}")
print(f"Vocabulary size: {tokenizer.vocab_size}")

Using device: cuda
Model loaded: bert-base-uncased
Vocabulary size: 30522


Define Pre-Process function to tokenize the data

In [53]:
def preprocess_function(examples):
    """
    Preprocess the dataset for training

    Args:
        examples: Dataset examples from SQuAD

    Returns:
        Tokenized inputs with start and end positions
    """
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=384,
        truncation="only_second",
        return_offsets_mapping=True,
        padding="max_length",
    )

    offset_mapping = inputs.pop("offset_mapping")
    answers = examples["answers"]
    start_positions = []
    end_positions = []

    for i, offset in enumerate(offset_mapping):
        answer = answers[i]
        start_char = answer["answer_start"][0]
        end_char = answer["answer_start"][0] + len(answer["text"][0])
        sequence_ids = inputs.sequence_ids(i)

        # Find the start and end of the context
        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        # If the answer is not fully inside the context, label it (0, 0)
        if offset[context_start][0] > end_char or offset[context_end][1] < start_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            # Otherwise it's the start and end token positions
            idx = context_start
            while idx <= context_end and offset[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            idx = context_end
            while idx >= context_start and offset[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

print("Preprocessing function defined!")

Preprocessing function defined!


In [54]:
dataset = load_dataset("squad")
dataset.shape

{'train': (87599, 5), 'validation': (10570, 5)}

In [56]:
num_train_examples = 2000  # Use subset for demonstration
num_eval_examples = 200
num_epochs = 2
learning_rate = 3e-5
batch_size = 16

# Use a subset for demonstration (full dataset takes longer)
train_dataset = dataset["train"].shuffle(seed=42).select(range(num_train_examples))
eval_dataset = dataset["validation"].shuffle(seed=42).select(range(min(num_eval_examples, len(dataset["validation"]))))

print(f"Training examples: {len(train_dataset)}")
print(f"Validation examples: {len(eval_dataset)}")


Training examples: 2000
Validation examples: 200


In [57]:
print("\nSample training example:")
sample = train_dataset[5]
print(f"Context: {sample['context'][:200]}...")
print(f"Question: {sample['question']}")
print(f"Answer: {sample['answers']['text'][0]}")


Sample training example:
Context: Most former British colonies and protectorates are among the 53 member states of the Commonwealth of Nations, a non-political, voluntary association of equal members, comprising a population of around...
Question: What is the population of the Commonwealth?
Answer: 2.2 billion


Tokenize training and evaluation dataset

In [58]:
tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

print("Preprocessing validation dataset...")
tokenized_eval = eval_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=eval_dataset.column_names
)

print("Dataset preprocessing completed!")
print(f"Training dataset size: {len(tokenized_train)}")
print(f"Validation dataset size: {len(tokenized_eval)}")


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Preprocessing validation dataset...


Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Dataset preprocessing completed!
Training dataset size: 2000
Validation dataset size: 200


Initilize Training arguements

In [59]:
training_args = TrainingArguments(
    output_dir="./bert-qa-finetuned",
    eval_strategy="epoch",
    learning_rate=learning_rate,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=num_epochs,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to='none',  # Disable wandb logging
)


Initilize and train Trainer

In [60]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    tokenizer=tokenizer,
    data_collator=default_data_collator,
)

print("Trainer initialized successfully!")
print(f"Training will run for {num_epochs} epochs")
print(f"Batch size: {batch_size}")
print(f"Learning rate: {learning_rate}")

Trainer initialized successfully!
Training will run for 2 epochs
Batch size: 16
Learning rate: 3e-05


In [61]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.257100,2.225671
2,2.033800,1.965382


TrainOutput(global_step=250, training_loss=2.858276672363281, metrics={'train_runtime': 317.444, 'train_samples_per_second': 12.601, 'train_steps_per_second': 0.788, 'total_flos': 783890270208000.0, 'train_loss': 2.858276672363281, 'epoch': 2.0})

In [62]:
trainer.save_model("./bert-qa-finetuned")
tokenizer.save_pretrained("./bert-qa-finetuned")

('./bert-qa-finetuned/tokenizer_config.json',
 './bert-qa-finetuned/special_tokens_map.json',
 './bert-qa-finetuned/vocab.txt',
 './bert-qa-finetuned/added_tokens.json',
 './bert-qa-finetuned/tokenizer.json')

In [ ]:
def load_finetuned_model(model_path="./bert-qa-finetuned"):
    """
    Load a fine-tuned model if available
    """
    global model, tokenizer
    try:
        model = AutoModelForQuestionAnswering.from_pretrained(model_path)
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        model.to(device)
        print(f"Fine-tuned model loaded from {model_path}")
        return True
    except Exception as e:
        print(f"Could not load fine-tuned model: {e}")
        print("Using base model instead")
        return False

# Try to load fine-tuned model
model_loaded = load_finetuned_model()

In [63]:
def answer_question(question, context, max_answer_length=30):
    """
    Answer a question given context using the BERT model

    Args:
        question (str): Question to answer
        context (str): Context containing the answer
        max_answer_length (int): Maximum length of answer

    Returns:
        dict: Answer with confidence score and positions
    """
    # Tokenize inputs
    inputs = tokenizer(
        question,
        context,
        max_length=512,
        truncation=True,
        return_tensors="pt",
        padding=True
    )

    # Move inputs to device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Get model predictions
    with torch.no_grad():
        outputs = model(**inputs)

    # Get start and end logits
    start_logits = outputs.start_logits
    end_logits = outputs.end_logits

    # Find the most likely start and end positions
    start_idx = torch.argmax(start_logits, dim=1).item()
    end_idx = torch.argmax(end_logits, dim=1).item()

    # Ensure end comes after start and within reasonable length
    if end_idx < start_idx:
        end_idx = start_idx
    if end_idx - start_idx > max_answer_length:
        end_idx = start_idx + max_answer_length

    # Calculate confidence score
    start_score = torch.softmax(start_logits, dim=1)[0][start_idx].item()
    end_score = torch.softmax(end_logits, dim=1)[0][end_idx].item()
    confidence = (start_score + end_score) / 2

    # Extract answer from tokens
    input_ids = inputs["input_ids"][0]
    answer_tokens = input_ids[start_idx:end_idx + 1]
    answer = tokenizer.decode(answer_tokens, skip_special_tokens=True)

    return {
        "answer": answer.strip(),
        "confidence": confidence,
        "start_position": start_idx,
        "end_position": end_idx
    }

In [64]:
# Sample contexts and questions for testing
test_cases = [
    {
        "context": """
        The Amazon rainforest, also known as Amazonia, is a moist broadleaf tropical rainforest
        in the Amazon biome that covers most of the Amazon basin of South America. The basin is
        6.7 million square kilometers, of which 5.5 million square kilometers are covered by the
        rainforest. The Amazon represents over half of the planet's remaining rainforests and
        comprises the largest and most biodiverse tract of tropical rainforest in the world.
        """,
        "question": "How large is the Amazon basin?"
    }]



In [65]:
for i, test_case in enumerate(test_cases, 1):
    print(f"\n--- Test Case {i} ---")
    print(f"Context: {test_case['context'][:100]}...")
    print(f"Question: {test_case['question']}")

    result = answer_question(test_case['question'], test_case['context'])

    print(f"Answer: {result['answer']}")
    print(f"Confidence: {result['confidence']:.4f}")
    print("-" * 40)



--- Test Case 1 ---
Context: 
        The Amazon rainforest, also known as Amazonia, is a moist broadleaf tropical rainforest 
  ...
Question: How large is the Amazon basin?
Answer: 6. 7 million square kilometers
Confidence: 0.6747
----------------------------------------


In [66]:
def interactive_qa():
    """
    Interactive question answering function
    """
    print("\n" + "=" * 60)
    print("Interactive BERT Question Answering")
    print("=" * 60)
    print("Enter your context and questions below.")
    print("Type 'quit' to exit at any time.")

    while True:
        print("\n" + "-" * 40)
        context = input("Enter context: ").strip()
        if context.lower() == 'quit':
            break

        question = input("Enter question: ").strip()
        if question.lower() == 'quit':
            break

        if context and question:
            print("\nProcessing...")
            result = answer_question(question, context)
            print(f"\nAnswer: {result['answer']}")
            print(f"Confidence: {result['confidence']:.4f}")

            # Show additional details
            print(f"Start position: {result['start_position']}")
            print(f"End position: {result['end_position']}")
        else:
            print("Please provide both context and question.")

    print("\nInteractive session ended!")

Test some interactive Questions and Answers

In [67]:
interactive_qa()


Interactive BERT Question Answering
Enter your context and questions below.
Type 'quit' to exit at any time.

----------------------------------------
Enter context: Crude ideas and designs of automobiles can be traced back to ancient and medieval times.[1][2] In 1649, Hans Hautsch of Nuremberg built a clockwork-driven carriage.[1][3] In 1672, a small-scale steam-powered vehicle was created by Ferdinand Verbiest;[4] the first steam-powered automobile capable of human transportation was built by Nicolas-Joseph Cugnot in 1769.[5][6] Inventors began to branch out at the start of the 19th century, creating the de Rivaz engine, one of the first internal combustion engines,[7] and an early electric motor.[8] Samuel Brown later tested the first industrially applied internal combustion engine in 1826. Only two of these were made. 
Enter question: when was internal combustion engine tested

Processing...

Answer: samuel brown
Confidence: 0.5862
Start position: 144
End position: 145

----------